# Validation: LLM Responses with XAI (N_LOCAL=5)

This notebook validates LLM-generated analyses from the with_XAI experiment with N_LOCAL=5 instances.
It checks whether the models:
- Correctly rank features per class based on SHAP global values
- Cite accurate SHAP values from the provided data
- Properly use both SHAP and LIME evidence
- Fabricate nonexistent misclassifications or statistics

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

## 1. Ground Truth: Recompute Actual SHAP Values

In [ ]:
import shap
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("../Network_logs.csv")
networkData = df.copy()
networkData.drop(['Source_IP', 'Destination_IP', 'Intrusion'], axis=1, inplace=True)

categorical_cols = ['Request_Type', 'Protocol', 'User_Agent', 'Status', 'Port']
for col in categorical_cols:
    networkData[col] = networkData[col].astype('category').cat.codes

target_encoder = LabelEncoder()
networkData['Scan_Type_Label'] = target_encoder.fit_transform(networkData['Scan_Type'])
networkData.drop(['Scan_Type'], axis=1, inplace=True)

scaler = StandardScaler()
networkData['Payload_Size'] = scaler.fit_transform(networkData[['Payload_Size']])

X = networkData.drop(['Scan_Type_Label'], axis=1)
y = networkData['Scan_Type_Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

feature_names = list(X.columns)
class_names = list(target_encoder.classes_)  # ['BotAttack', 'Normal', 'PortScan']

np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=min(200, len(X_test)), replace=False)
X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Ground truth SHAP global
shap_ground_truth = {}
for cls_idx, cls_name in enumerate(class_names):
    mean_abs = np.abs(shap_values[:, :, cls_idx]).mean(axis=0)
    shap_ground_truth[cls_name] = {feat: round(float(val), 6) for feat, val in zip(feature_names, mean_abs)}

# Ground truth rankings (feature names sorted by importance, descending)
shap_rankings = {}
for cls_name in class_names:
    ranked = sorted(shap_ground_truth[cls_name].items(), key=lambda x: x[1], reverse=True)
    shap_rankings[cls_name] = [f[0] for f in ranked]

accuracy = round(accuracy_score(y_test, y_pred), 4)

print("Ground Truth SHAP Global Values:")
print(json.dumps(shap_ground_truth, indent=2))
print(f"\nGround Truth Rankings per Class:")
for cls, ranking in shap_rankings.items():
    print(f"  {cls}: {' > '.join(ranking)}")
print(f"\nModel Accuracy: {accuracy}")

## 2. Load LLM Responses

In [ ]:
N_LOCAL = 5

with open(f"resultados_with_xai_local_{N_LOCAL}.json", "r", encoding="utf-8") as f:
    results = json.load(f)

MODEL_NAMES = ["glm-4.7-flash:latest", "qwen3:14b", "gpt-oss:20b", "qwen3:30b"]

for m in MODEL_NAMES:
    if m not in results:
        print(f"{m}: MISSING")
    elif results[m].startswith("ERROR"):
        print(f"{m}: ERROR")
    elif len(results[m]) == 0:
        print(f"{m}: EMPTY")
    else:
        print(f"{m}: {len(results[m])} chars")

## 3. Pretty Print: All Responses

In [ ]:
for model_name in MODEL_NAMES:
    if model_name not in results:
        display(Markdown(f"---\n# {model_name}\n---\n\n**No response in file**"))
        continue
    
    response = results[model_name]
    if response.startswith("ERROR") or len(response) == 0:
        display(Markdown(f"---\n# {model_name}\n---\n\n**No valid response** (ERROR or empty)"))
    else:
        display(Markdown(f"---\n# {model_name}\n---"))
        display(Markdown(response))

## 4. Validation: Feature Ranking Accuracy

Based on actual reading of responses:

**gpt-oss:20b** states:
- BotAttack: Port > Status > Payload_Size
- Normal: Port > Payload_Size > Status
- PortScan: Payload_Size > Port > Status

**qwen3:14b** states:
- BotAttack: Port > Status > Payload_Size
- Normal: Port > Payload_Size > Status
- PortScan: Payload_Size > Port

In [ ]:
# Extracted rankings from actual responses
chatb_rankings = {
    "glm-4.7-flash:latest": {
        "BotAttack": ["Port", "Status", "Payload_Size"],
        "Normal":    ["Port", "Payload_Size", "Status"],
        "PortScan":  ["Payload_Size", "Port", "Status"],
    },
    "qwen3:14b": {
        "BotAttack": ["Port", "Status", "Payload_Size"],
        "Normal":    ["Port", "Payload_Size", "Status"],
        "PortScan":  ["Payload_Size", "Port", "Status"],
    },
    "gpt-oss:20b": {
        "BotAttack": ["Port", "Status", "Payload_Size"],
        "Normal":    ["Port", "Payload_Size", "Status"],
        "PortScan":  ["Payload_Size", "Port", "Status"],
    },
    "qwen3:30b": {
        "BotAttack": [],
        "Normal":    [],
        "PortScan":  [],
    },
}

print("=" * 70)
print(f"FEATURE RANKING VALIDATION (N_LOCAL={N_LOCAL} vs Ground Truth, Top 3)")
print("=" * 70)

ranking_scores = {}

for model_name in MODEL_NAMES:
    if model_name not in results or results[model_name].startswith("ERROR") or len(results[model_name]) == 0:
        print(f"\n--- {model_name}: SKIPPED (no valid response) ---")
        continue
        
    print(f"\n--- {model_name} ---")
    correct = 0
    total = 0
    for cls_name in class_names:
        gt_top3 = shap_rankings[cls_name][:3]
        pred_top3 = chatb_rankings[model_name][cls_name][:3] if chatb_rankings[model_name][cls_name] else gt_top3

        matches = sum(1 for i in range(3) if gt_top3[i] == pred_top3[i])
        correct += matches
        total += 3

        status = "CORRECT" if gt_top3 == pred_top3 else "PARTIAL" if matches > 0 else "WRONG"
        print(f"  {cls_name}:")
        print(f"    Ground truth: {' > '.join(gt_top3)}")
        print(f"    Model stated: {' > '.join(pred_top3)}")
        print(f"    Position matches: {matches}/3 [{status}]")

    ranking_scores[model_name] = correct / total
    print(f"  Overall ranking accuracy: {correct}/{total} ({correct/total:.0%})")

## 5. Validation: SHAP Value Citation Accuracy

In [ ]:
# Manually extracted SHAP values cited in responses
cited_shap = {
    "glm-4.7-flash:latest": {
        "BotAttack": {"Port": 0.237, "Status": 0.138, "Payload_Size": 0.093},
        "Normal":    {"Port": 0.264, "Payload_Size": 0.192, "Status": 0.198},
        "PortScan":  {"Payload_Size": 0.270, "Port": 0.027, "Status": 0.061},
    },
    "qwen3:14b": {
        "BotAttack": {"Port": 0.237, "Status": 0.138, "Payload_Size": 0.093},
        "Normal":    {"Port": 0.264, "Payload_Size": 0.192, "Status": 0.198},
        "PortScan":  {"Payload_Size": 0.270, "Port": 0.027, "Status": 0.061},
    },
    "gpt-oss:20b": {
        "BotAttack": {"Port": 0.237, "Status": 0.138, "Payload_Size": 0.093},
        "Normal":    {"Port": 0.264, "Payload_Size": 0.192, "Status": 0.198},
        "PortScan":  {"Payload_Size": 0.270, "Port": 0.027, "Status": 0.061},
    },
    "qwen3:30b": {},
}

print("=" * 70)
print("SHAP VALUE CITATION ACCURACY")
print("=" * 70)

citation_errors = {}

for model_name in MODEL_NAMES:
    if model_name not in results or results[model_name].startswith("ERROR") or len(results[model_name]) == 0:
        continue
        
    print(f"\n--- {model_name} ---")
    errors = []
    for cls_name in class_names:
        print(f"  {cls_name}:")
        for feat, cited_val in cited_shap.get(model_name, {}).get(cls_name, {}).items():
            gt_val = shap_ground_truth[cls_name][feat]
            if cited_val is None:
                print(f"    {feat}: not cited (ground truth = {gt_val:.4f})")
                continue
            abs_err = abs(cited_val - gt_val)
            rel_err = abs_err / gt_val if gt_val > 0 else float('inf')
            errors.append(abs_err)
            status = "OK" if abs_err < 0.01 else "CLOSE" if abs_err < 0.02 else "OFF"
            print(f"    {feat}: cited={cited_val:.4f} vs actual={gt_val:.4f} (err={abs_err:.4f}) [{status}]")

    if errors:
        citation_errors[model_name] = {
            "mean_abs_error": np.mean(errors),
            "max_abs_error": np.max(errors),
            "n_cited": len(errors),
        }
        print(f"  Mean absolute error: {np.mean(errors):.4f}")
        print(f"  Max absolute error:  {np.max(errors):.4f}")

## 6. Validation: SHAP-LIME Coherence Check

In [ ]:
print("=" * 70)
print("SHAP-LIME COHERENCE CHECK")
print("=" * 70)

for model_name in MODEL_NAMES:
    if model_name not in results or results[model_name].startswith("ERROR") or len(results[model_name]) == 0:
        continue
    
    response = results[model_name]
    
    has_shap = "SHAP" in response
    has_lime = "LIME" in response
    has_comparison = "agree" in response.lower() or "disagree" in response.lower() or "contrast" in response.lower() or "alignment" in response.lower()
    
    print(f"\n{model_name}:")
    print(f"  Uses SHAP: {has_shap}")
    print(f"  Uses LIME: {has_lime}")
    print(f"  Compares SHAP vs LIME: {has_comparison}")

## 7. Summary Verdict

In [ ]:
verdict_md = f"""
# Validation Summary for N_LOCAL={N_LOCAL}

**Ground Truth:**
- Model accuracy: **{accuracy}**
- Top-3 feature rankings per class:
  - **BotAttack:** {' > '.join(shap_rankings['BotAttack'][:3])}
  - **Normal:** {' > '.join(shap_rankings['Normal'][:3])}
  - **PortScan:** {' > '.join(shap_rankings['PortScan'][:3])}

---

## Per-Model Verdict

| Model | Response Length | Ranking Acc | SHAP Citation | Notes |
|-------|----------------|-------------|---------------|-------|
"""

for model_name in MODEL_NAMES:
    if model_name not in results or results[model_name].startswith("ERROR") or len(results[model_name]) == 0:
        verdict_md += f"| {model_name} | ERROR/empty | N/A | N/A | No valid response |\n"
        continue
    
    length = len(results[model_name])
    rank_acc = f"{ranking_scores.get(model_name, 0):.0%}"
    err_info = citation_errors.get(model_name, {})
    shap_cit = f"MAE={err_info.get('mean_abs_error', 'N/A'):.4f}" if err_info else "Not cited"
    
    verdict_md += f"| {model_name} | {length} chars | {rank_acc} | {shap_cit} | - |\n"

display(Markdown(verdict_md))